# Bronze Layer
## Load data

In [0]:
# Create schema to Adult Income Raw (Bronze Layer)
spark.sql("""
CREATE SCHEMA IF NOT EXISTS adult_income.adult_income_raw
""")

In [0]:
from sklearn.datasets import fetch_openml

# 1. Carregar dataset Adult Income
adult = fetch_openml(name='adult', version=2, as_frame=True)
df = adult.frame.copy()

# 2. Convert pandas DataFrame to Spark DataFrame
df_bronze = spark.createDataFrame(df)

# 3. Write the transformed data to the Bronze layer
bronze_data_path = "adult_income.adult_income_raw.ad_inc_raw"
df_bronze.write.format("delta").mode("overwrite").saveAsTable(bronze_data_path)

# Register the DataFrame as a temporary view so we can run SQL queries
df_bronze.createOrReplaceTempView("ad_inc_raw")

# Example SQL query on the temporary view
result_df = spark.sql("SELECT COUNT(*) as trip_count FROM ad_inc_raw")

# Silver Layer
## Data Preparation

In [0]:
# Create schema to Adult Income Raw (Bronze Layer)
spark.sql("""
CREATE SCHEMA IF NOT EXISTS adult_income.adult_income_silver
""")

In [0]:
# Carregar dados da camada Bronze (ad_inc_raw)
df_raw = spark.table("adult_income.adult_income_raw.ad_inc_raw").toPandas()

# Convert string columns to categorical dtype
for col in ['workclass', 'occupation', 'native-country']:
    df_raw[col] = df_raw[col].astype('category')

# 1) Adicionar a categoria 'Unemployed' apenas se ela ainda não existir
if 'Unemployed' not in df_raw['workclass'].cat.categories:
    df_raw['workclass'] = df_raw['workclass'].cat.add_categories(['Unemployed'])

# 1.1) Preencher os valores nulos
df_raw['workclass'] = df_raw['workclass'].fillna('Unemployed')

# 2) Tratamento de missing values para 'occupation' e 'native-country'
for col in ['occupation', 'native-country']:
    if 'missing' not in df_raw[col].cat.categories:
        df_raw[col] = df_raw[col].cat.add_categories(['missing'])
    df_raw[col] = df_raw[col].fillna('missing')

# Convert pandas DataFrame to Spark DataFrame
df_silver = spark.createDataFrame(df_raw)

# Write the transformed data to the Silver layer
silver_data_path = "adult_income.adult_income_silver.ad_inc_silver"
df_silver.write.format("delta").mode("overwrite").saveAsTable(silver_data_path)

# Register the DataFrame as a temporary view so we can run SQL queries
df_silver.createOrReplaceTempView("ad_inc_silver")

print("Silver layer processing completed.")

# Gold Layer
## Feature Engineering

In [0]:
# Create schema to Adult Income Raw (Bronze Layer)
spark.sql("""
CREATE SCHEMA IF NOT EXISTS adult_income.adult_income_gold
""")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, when
from itertools import chain

# Load data from Silver layer
df_silver = spark.table("adult_income.adult_income_silver.ad_inc_silver")

# Create feature engineered DataFrame using PySpark
df_feature = df_silver \
    .withColumn('is_capital_gain', when(col('capital-gain') > 0, 1).otherwise(0)) \
    .drop('capital-gain') \
    .withColumn('is_private', when(col('workclass') == 'Private', 1).otherwise(0)) \
    .withColumn('is_not_private', when(col('workclass') != 'Private', 1).otherwise(0)) \
    .withColumn('is_white', when(col('race') == 'White', 1).otherwise(0)) \
    .withColumn('is_not_white', when(col('race') != 'White', 1).otherwise(0)) \
    .withColumn('is_male', when(col('sex') == 'Male', 1).otherwise(0)) \
    .withColumn('is_female', when(col('sex') == 'Female', 1).otherwise(0)) \
    .withColumn('is_occupation_missing', when(col('occupation') == 'missing', 1).otherwise(0)) \
    .withColumn('is_native-country_missing', when(col('native-country') == 'missing', 1).otherwise(0)) \
    .withColumn('is_from_United-States', when(col('native-country') == 'United-States', 1).otherwise(0)) \
    .withColumn('is_capital_loss', when(col('capital-loss') > 0, 1).otherwise(0)) \
    .withColumn('is_married', when(col('marital-status') == 'Married-civ-spouse', 1).otherwise(0)) \
    .withColumn('is_married_and_not', when((col('marital-status') == 'Married-civ-spouse') | (col('marital-status') == 'Never-married'), 1).otherwise(0)) \
    .withColumn('is_married_notmarried_divorced', when((col('marital-status') == 'Married-civ-spouse') | (col('marital-status') == 'Never-married') | (col('marital-status') == 'Divorced'), 1).otherwise(0)) \
    .withColumn('is_husband', when(col('relationship') == 'Husband', 1).otherwise(0)) \
    .withColumn('is_wife', when(col('relationship') == 'Wife', 1).otherwise(0)) \
    .withColumn('is_husb_and_not_in_fam', when((col('relationship') == 'Husband') & (col('relationship') != 'Not-in-family'), 1).otherwise(0)) \
    .withColumn('edu_x_hours', col('education-num') * col('hours-per-week'))

# sex: mantido como ordinal (categoria binária, sem risco de distância falsa)
df_feature = df_feature.withColumn('sex_ord',
    when(col('sex') == 'Male', 1)
    .when(col('sex') == 'Female', 2)
    .otherwise(None)
)

# Write to Gold layer
gold_data_path = "adult_income.adult_income_gold.ad_inc_gold"
df_feature.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(gold_data_path)

print("Gold layer processing completed.")
print(f"Data saved to: {gold_data_path}")